# server.py: The knowledge base server

**`server.py` offers one tool, `get_knowledge_base`, which reads five company-policy questions and answers from `data/kb.json` and returns them as one text.** An LLM calls it when a question needs company information. The server runs over stdio only, so the client notebooks start it themselves.

Read `../2-server-stdio-http-sse/server.ipynb` first. This notebook covers only what's new.

## The whole file

<div style="font-size: 0.85em">

```python
import json
import os

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Knowledge Base")


@mcp.tool()
def get_knowledge_base() -> str:
    """Retrieve the company knowledge base, with every question and answer about company policies."""
    folder = os.path.dirname(__file__)
    kb_path = os.path.join(folder, "data", "kb.json")

    with open(kb_path) as f:
        kb_data = json.load(f)

    text = "Here is the retrieved knowledge base:\n\n"
    for item in kb_data:
        text += "Q: " + item["question"] + "\n"
        text += "A: " + item["answer"] + "\n\n"

    return text


if __name__ == "__main__":
    mcp.run(transport="stdio")
```

</div>

## 1. The data

`json.load(f)` reads a JSON file into Python: JSON lists become Python lists, and JSON objects become dicts.

In [1]:
import json

with open("data/kb.json") as f:
    kb_data = json.load(f)

print(type(kb_data))
print(len(kb_data))
print(kb_data[0])

<class 'list'>
5
{'question': "What is our company's vacation policy?", 'answer': 'Full-time employees are entitled to 20 paid vacation days per year. Vacation days can be taken after completing 6 months of employment. Unused vacation days can be carried over to the next year up to a maximum of 5 days. Vacation requests should be submitted at least 2 weeks in advance through the HR portal.'}


A list of five dicts, each with a `"question"` and an `"answer"`.

## 2. Finding the file from anywhere

`__file__` is a variable Python sets in every file to that file's own path. `os.path.dirname(__file__)` keeps only its folder, and `os.path.join(folder, "data", "kb.json")` adds the rest of the path with the right separator for Windows or Linux.

So the server finds `kb.json` next to itself, whichever folder it was started from. That matters because a host like Claude Desktop starts its servers from its own folder, not from yours.

## 3. What the tool returns

`@mcp.tool()` registers the function and gives it back unchanged, so you can still call it as plain Python:

In [2]:
import server

text = server.get_knowledge_base()
print(text)

Here is the retrieved knowledge base:

Q: What is our company's vacation policy?
A: Full-time employees are entitled to 20 paid vacation days per year. Vacation days can be taken after completing 6 months of employment. Unused vacation days can be carried over to the next year up to a maximum of 5 days. Vacation requests should be submitted at least 2 weeks in advance through the HR portal.

Q: How do I request a new software license?
A: To request a new software license, please submit a ticket through the IT Service Desk portal. Include the software name, version, and business justification. Standard software licenses are typically approved within 2 business days. For specialized software, approval may take up to 5 business days and may require department head approval.

Q: What is our remote work policy?
A: Our company follows a hybrid work model. Employees can work remotely up to 3 days per week. Remote work days must be coordinated with your team and approved by your direct manager

This is exactly the text the model receives as the tool's result: All five pairs, every time. The tool doesn't search. The model reads everything and picks the answer it needs. That's fine for five entries, but a real knowledge base would take the question as an argument and return only the matching parts, which is what RAG does.

## 4. The schema the model sees

In [3]:
tools = await server.mcp.list_tools()
print(tools[0].model_dump_json(indent=2, exclude_none=True))

{
  "name": "get_knowledge_base",
  "description": "Retrieve the company knowledge base, with every question and answer about company policies.",
  "inputSchema": {
    "properties": {},
    "title": "get_knowledge_baseArguments",
    "type": "object"
  },
  "outputSchema": {
    "properties": {
      "result": {
        "title": "Result",
        "type": "string"
      }
    },
    "required": [
      "result"
    ],
    "title": "get_knowledge_baseOutput",
    "type": "object"
  }
}


- **`inputSchema`:** It has no `properties`, because the tool takes no arguments. That's why the model calls it with `{}`.
- **`description`:** The docstring. It's the only thing that tells the model when to call the tool, so it names what's inside: Questions and answers about company policies.

## 5. Errors

There's no `try/except` in the tool, because FastMCP already handles errors. If the tool raises an exception, for example because `kb.json` is missing, FastMCP catches it and sends the client a result with `isError: true` and the error message.

## Say it out loud

> "The server exposes one tool, `get_knowledge_base`, that returns the company's policy Q&As from a JSON file. It takes no arguments, so its input schema is empty, and its docstring is the description the model uses to decide when to call it. It builds the file path from `__file__`, because a host may start it from any folder. For a large knowledge base, I'd make the tool take the question and return only the matching chunks, which is RAG."